# Parallel Web Scraping with Browser-Harness + Playwright Workspaces

This notebook demonstrates how to:
1. Create a Playwright Workspace (PWW) on Azure
2. Connect browser-harness to the PWW remote CDP endpoint
3. Spawn 10+ parallel browser sessions for web scraping
4. Use LiveView for real-time debuggability

**Target**: Scrape product data from [books.toscrape.com](http://books.toscrape.com) across multiple category pages simultaneously.

## Section 1: Prerequisites & Setup

Ensure you have:
- Azure CLI authenticated (`az login`)
- browser-harness installed (`git clone https://github.com/browser-use/browser-harness && cd browser-harness && uv tool install -e .`)
- Dependencies installed (`pip install -r requirements.txt`)

In [ ]:
import os
import json
import uuid
import subprocess
from datetime import datetime, timedelta, timezone
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.mgmt.playwright import PlaywrightMgmtClient
from azure.mgmt.playwright.models import PlaywrightWorkspace, PlaywrightWorkspaceProperties
from azure.developer.playwright import PlaywrightClient

load_dotenv()

# Configuration
SUBSCRIPTION_ID = os.environ["SUBSCRIPTION_ID"]
RESOURCE_GROUP = os.environ["RESOURCE_GROUP"]
LOCATION = os.environ.get("LOCATION", "eastus")
PLAYWRIGHT_WORKSPACE_NAME = os.environ["PLAYWRIGHT_WORKSPACE_NAME"]

credential = DefaultAzureCredential()
print("✅ Configuration loaded")

## Section 2: Create Playwright Workspace (PWW)

This creates a managed Playwright Workspace on Azure that provides cloud-hosted browsers.
Skip this cell if your workspace already exists.

In [ ]:
# Create or get the Playwright Workspace
pw_mgmt = PlaywrightMgmtClient(credential, SUBSCRIPTION_ID)

print(f"Creating Playwright Workspace: {PLAYWRIGHT_WORKSPACE_NAME}...")
workspace = pw_mgmt.playwright_workspaces.begin_create_or_update(
    resource_group_name=RESOURCE_GROUP,
    playwright_workspace_name=PLAYWRIGHT_WORKSPACE_NAME,
    resource=PlaywrightWorkspace(
        location=LOCATION,
        properties=PlaywrightWorkspaceProperties(local_auth="Enabled"),
    ),
).result()

workspace_id = workspace.properties.workspace_id
dataplane_uri = workspace.properties.dataplane_uri
base_url = f"{urlparse(dataplane_uri).scheme}://{urlparse(dataplane_uri).netloc}"

print(f"✅ Workspace ready: {workspace_id}")
print(f"   Dataplane: {base_url}")

In [ ]:
# Create an access token for the workspace
pw_client = PlaywrightClient(endpoint=base_url, credential=credential)

access_token_id = str(uuid.uuid4())
token = pw_client.access_tokens.create_or_replace(
    workspace_id=workspace_id,
    access_token_id=access_token_id,
    resource={
        "name": f"scraping-demo-{access_token_id[:8]}",
        "expiryAt": (datetime.now(timezone.utc) + timedelta(days=30)).isoformat()
    }
)

playwright_api_key = token.jwt_token
print("✅ Access token created (valid 30 days)")

## Section 3: Connect Browser-Harness to PWW Remote Endpoint

### The Connection Prompt

Paste the following prompt into your coding agent (Claude Code / Codex) to connect browser-harness to the PWW remote browser:

---

```text
Set up browser-harness to connect to my Playwright Workspaces remote browser.

Read install.md and SKILL.md first. Then connect to this Azure Playwright Service endpoint:

  SERVICE_URL=<paste your PWW service URL here>

Follow the two-step connection flow:
1. HTTP GET the SERVICE_URL (allow 60-90s for the browser to spin up). Parse the JSON response to extract the `sessionUrl` (a wss:// WebSocket URL).
2. Set BU_CDP_WS to the resolved sessionUrl in .env, then restart the daemon ONCE.

IMPORTANT:
- Do NOT kill or restart the daemon after the session is connected — the remote browser is destroyed when the WebSocket closes.
- Do NOT set shouldRedirect=true; use shouldRedirect=false and manually resolve the sessionUrl.
- The cold start takes 30-90s. Use a generous timeout on the initial HTTP GET.
- After connecting, verify with: browser-harness <<'PY'\nprint(page_info())\nPY

Once connected, confirm with a screenshot that the remote browser is alive.
```

---

### Programmatic Connection (alternative)

If you prefer to connect programmatically instead of via the coding agent prompt:

In [ ]:
import urllib.request

# Build the PWW service URL
service_url = (
    f"https://{urlparse(dataplane_uri).netloc}"
    f"/playwrightworkspaces/{workspace_id}/browsers"
    f"?playwrightVersion=cdp&shouldRedirect=false"
    f"&accessKey={playwright_api_key}"
)

print("Resolving remote browser session (30-90s cold start)...")

# Step 1: HTTP GET to provision the browser and get the CDP WebSocket URL
resp = urllib.request.urlopen(service_url, timeout=120)
data = json.loads(resp.read())
cdp_ws_url = data["sessionUrl"]

print(f"✅ Remote browser provisioned")
print(f"   CDP WebSocket: {cdp_ws_url[:80]}...")

# Step 2: Write to .env so browser-harness picks it up
env_path = os.path.join(os.path.dirname(os.path.abspath('.')), 'browser-harness', '.env')
# Or set it directly for this session:
os.environ["BU_CDP_WS"] = cdp_ws_url

print("\n⚠️  IMPORTANT: Do NOT restart the daemon after this point.")
print("   The remote browser is destroyed when the WebSocket closes.")

In [ ]:
# Verify connection
# NOTE: browser-harness uses -c flag for script execution
result = subprocess.run(
    ["browser-harness", "-c", "print(page_info())"],
    capture_output=True, text=True, timeout=30,
    env={**os.environ, "BU_CDP_WS": cdp_ws_url}
)
print(result.stdout)
if result.returncode == 0:
    print("✅ Browser-harness connected to PWW remote browser")
else:
    print(f"❌ Connection failed: {result.stderr}")

## Section 4: Parallel Web Scraping (10+ Sessions)

We'll scrape product data from [books.toscrape.com](http://books.toscrape.com) — a public demo site designed for scraping practice.

Each parallel session scrapes a different category page.

In [ ]:
# Define the pages to scrape (one per parallel browser session)
CATEGORY_URLS = [
    "http://books.toscrape.com/catalogue/category/books/travel_2/index.html",
    "http://books.toscrape.com/catalogue/category/books/mystery_3/index.html",
    "http://books.toscrape.com/catalogue/category/books/historical-fiction_4/index.html",
    "http://books.toscrape.com/catalogue/category/books/sequential-art_5/index.html",
    "http://books.toscrape.com/catalogue/category/books/classics_6/index.html",
    "http://books.toscrape.com/catalogue/category/books/philosophy_7/index.html",
    "http://books.toscrape.com/catalogue/category/books/romance_8/index.html",
    "http://books.toscrape.com/catalogue/category/books/womens-fiction_9/index.html",
    "http://books.toscrape.com/catalogue/category/books/fiction_10/index.html",
    "http://books.toscrape.com/catalogue/category/books/childrens_11/index.html",
    "http://books.toscrape.com/catalogue/category/books/religion_12/index.html",
    "http://books.toscrape.com/catalogue/category/books/nonfiction_13/index.html",
]

print(f"Will scrape {len(CATEGORY_URLS)} category pages in parallel")

In [ ]:
# Scraping script template for each parallel browser session
# browser-harness uses -c flag: browser-harness -c "<script>"
SCRAPE_SCRIPT = (
    'import json\n'
    'new_tab("{url}")\n'
    'wait_for_load()\n'
    'books = js("Array.from(document.querySelectorAll(\'article.product_pod\')).map(el => ({{'
    'title: el.querySelector(\'h3 a\').getAttribute(\'title\'),'
    'price: el.querySelector(\'.price_color\').textContent,'
    'availability: el.querySelector(\'.availability\').textContent.trim(),'
    'rating: el.querySelector(\'p.star-rating\').className.replace(\'star-rating \', \'\')'
    '}}))"\n)\n'
    'category = js("document.querySelector(\'.page-header h1\').textContent")\n'
    'print(json.dumps({{"category": category, "books": books}}))'
)


def scrape_category(url, session_name):
    """Scrape a single category page using a dedicated browser session.
    
    Each session gets its own remote browser via a distinct BU_NAME.
    The daemon must connect immediately after provisioning — the session
    URL is ephemeral and expires quickly.
    """
    script = SCRAPE_SCRIPT.format(url=url)
    result = subprocess.run(
        ["browser-harness", "-c", script],
        capture_output=True, text=True, timeout=60,
        env={**os.environ, "BU_NAME": session_name, "BU_CDP_WS": cdp_ws_url}
    )
    if result.returncode == 0:
        # Parse the JSON output from the last print statement
        for line in result.stdout.strip().split('\n'):
            try:
                return json.loads(line)
            except json.JSONDecodeError:
                continue
    return {"error": result.stderr, "url": url}


print("Ready to scrape. Running parallel sessions...")

In [ ]:
# Execute parallel scraping across all categories
all_results = []

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {
        executor.submit(scrape_category, url, f"scraper-{i}"): url
        for i, url in enumerate(CATEGORY_URLS)
    }
    
    for future in as_completed(futures):
        url = futures[future]
        try:
            result = future.result()
            if "error" not in result:
                all_results.append(result)
                print(f"✅ {result['category']}: {len(result['books'])} books")
            else:
                print(f"❌ {url}: {result['error'][:100]}")
        except Exception as e:
            print(f"❌ {url}: {e}")

print(f"\n📊 Scraped {sum(len(r['books']) for r in all_results)} books from {len(all_results)} categories")

In [ ]:
# Aggregate into a DataFrame
rows = []
for result in all_results:
    for book in result["books"]:
        rows.append({
            "category": result["category"],
            "title": book["title"],
            "price": book["price"],
            "availability": book["availability"],
            "rating": book["rating"],
        })

df = pd.DataFrame(rows)
print(f"Total books scraped: {len(df)}")
df.head(15)

## Section 5: LiveView for Debuggability

LiveView lets you watch any browser session in real-time. The `LiveViewWatcher` polls for new sessions and auto-opens the viewer in your browser.

In [ ]:
from helpers.live_view_watcher import LiveViewWatcher

# Initialize the watcher
live_watcher = LiveViewWatcher(
    pw_client=pw_client,
    workspace_id=workspace_id,
    credential=credential,
    auth_token=playwright_api_key,
    auth_service_base=base_url,
)

# Start watching — when the next browser session is created, LiveView opens automatically
live_watcher.start()
print("👀 LiveView watcher active — will open viewer when a new session starts")

In [ ]:
# Run a single scraping task to trigger LiveView
demo_result = scrape_category(CATEGORY_URLS[0], "live-demo")
print(f"Scraped: {demo_result.get('category', 'unknown')}")

live_watcher.stop()
if live_watcher.session_id:
    print(f"\n✅ LiveView opened for session: {live_watcher.session_id}")
else:
    print("\nℹ️  No new session detected (session may have reused an existing one)")

## Section 6: Cleanup

Stop browser sessions and optionally delete the workspace.

In [ ]:
# List active sessions
sessions = list(pw_client.browser_sessions.list(workspace_id))
print(f"Active sessions: {len(sessions)}")
for s in sessions:
    print(f"  - {s.id} | {s.browser_type} | {s.status}")

In [ ]:
# Optional: Delete the workspace (uncomment to run)
# pw_mgmt.playwright_workspaces.begin_delete(
#     resource_group_name=RESOURCE_GROUP,
#     playwright_workspace_name=PLAYWRIGHT_WORKSPACE_NAME,
# ).result()
# print("✅ Workspace deleted")

print("Done! Your scraped data is in the 'df' DataFrame above.")